# RAG na Prática com LangChain + Groq

Este notebook demonstra um pipeline RAG completo da maneira mais simplificada possível.

**O que você vai ver:**
- Um LLM respondendo **sem** contexto (resposta genérica / incorreta)
- O mesmo LLM respondendo **com RAG** (resposta baseada nos documentos)

**Stack utilizada:**
| Componente | Ferramenta | Versão |
|---|---|---|
| LLM | Groq (Llama 3.1 8B) | llama-3.1-8b-instant |
| Embeddings | HuggingFace (multilíngue) | paraphrase-multilingual-MiniLM-L12-v2 |
| Vector Store | Chroma | 0.6.x |
| Orquestração | LangChain | 0.3.x |

> 🔑 Você precisa de uma chave Groq gratuita: https://console.groq.com

In [ ]:
# ── Instalação ─────────────────────────────────────────────────────
!pip install -q langchain==0.3.25 \
                langchain-groq \
                langchain-community \
                langchain-huggingface \
                chromadb \
                sentence-transformers

print("Pacotes instalados")

Pacotes instalados


In [ ]:
# ── Imports, chave e modelos ───────────────────────────────────────
import os
from getpass import getpass
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Cole sua chave em: https://console.groq.com → API Keys
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# LLM via Groq (rápido e gratuito)
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

# Embeddings multilíngues locais (não necessita API key, tamanho ideal para rodar no Colab)
embeddings = HuggingFaceEmbeddings(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

print("LLM e embeddings prontos")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

LLM e embeddings prontos


## Base de Conhecimento

Vamos criar documentos de uma empresa fictícia, a **Nexus Tecnologia Ltda.**

O LLM claramente não foi treinado nestes dados (uma vez que a empresa não existe), então qualquer resposta correta sobre eles **só pode vir do RAG**.

In [ ]:
# ── Documentos + Indexação ─────────────────────────────────────────

# Documentos internos da Nexus Tecnologia (fictícios)
documentos = [
    Document(
        page_content="""
        Política de Férias – Nexus Tecnologia Ltda.
        Funcionários têm direito a 25 dias corridos de férias por ano completado.
        O agendamento deve ser feito com 45 dias de antecedência via sistema RH-Nexus.
        É possível fracionar em até 3 períodos, sendo o menor deles de 5 dias corridos.
        Férias não usufruídas vencem após 24 meses do período aquisitivo.
        """,
        metadata={"fonte": "politica_ferias_v2.pdf", "departamento": "rh"}
    ),
    Document(
        page_content="""
        Política de Reembolso de Despesas – Nexus Tecnologia Ltda.
        Despesas de viagem e representação devem ser solicitadas em até 10 dias
        úteis após o evento, via formulário REE-07 no portal interno.
        O limite mensal por colaborador é de R$ 2.000,00.
        Despesas acima de R$ 500,00 exigem aprovação prévia do gestor direto.
        Reembolsos são processados toda primeira sexta-feira do mês.
        """,
        metadata={"fonte": "politica_reembolso_v3.pdf", "departamento": "financeiro"}
    ),
    Document(
        page_content="""
        Processo de Onboarding – Nexus Tecnologia Ltda.
        Novos colaboradores passam por 3 etapas: (1) Integração Institucional no
        primeiro dia, conduzida pelo RH; (2) Onboarding Técnico nos dias 2 a 5,
        com o time de TI; (3) Acompanhamento com o gestor nas semanas 2 e 3.
        O mentor designado permanece disponível por 90 dias.
        Acesso aos sistemas é liberado em até 4 horas úteis após a admissão.
        """,
        metadata={"fonte": "processo_onboarding_v1.pdf", "departamento": "rh"}
    ),
    Document(
        page_content="""
        Benefícios – Nexus Tecnologia Ltda.
        Vale-refeição: R$ 45,00 por dia útil (cartão Alelo).
        Plano de saúde: Unimed Nacional, cobertura nacional, sem coparticipação.
        Auxílio home office: R$ 150,00/mês para colaboradores em regime remoto.
        Gympass: plano prata disponível a partir de 6 meses de empresa.
        Day off de aniversário: folga remunerada no mês de aniversário.
        """,
        metadata={"fonte": "beneficios_2024.pdf", "departamento": "rh"}
    ),
]

# Indexar no Chroma (em memória, sem necessidade de salvar em disco)
vectorstore = Chroma.from_documents(
    documents=documentos,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"{len(documentos)} documentos indexados")

# Teste rápido do retriever
teste = retriever.invoke("quantos dias de férias")
print(f"\nTeste do retriever → {len(teste)} chunk(s) recuperado(s):")
print(f"  └── {teste[0].metadata['fonte']}")

4 documentos indexados

Teste do retriever → 2 chunk(s) recuperado(s):
  └── politica_ferias_v2.pdf


## Pipeline RAG

Montamos a chain com **LangChain Expression Language (LCEL)**:

```
[pergunta]
    ├── retriever  → busca os chunks relevantes
    └── passthrough → mantém a pergunta original
           ↓
        prompt  → monta pergunta + contexto
           ↓
          llm   → gera a resposta
           ↓
        parser  → extrai o texto final
```

In [ ]:
# ── CÉLULA 4 · Chain RAG ───────────────────────────────────────────────────────

# --- Prompt do sistema ---
prompt = ChatPromptTemplate.from_template("""
Você é o assistente interno da Nexus Tecnologia.
Responda de forma objetiva usando APENAS os documentos abaixo.
Se a resposta não estiver nos documentos, diga exatamente:
"Não encontrei essa informação nos documentos disponíveis."

Documentos:
{context}

Pergunta: {pergunta}
""")

# --- Função auxiliar: formata chunks com citação de fonte ---
def formatar_docs(docs):
    partes = []
    for doc in docs:
        fonte = doc.metadata.get("fonte", "desconhecida")
        partes.append(f"[Fonte: {fonte}]\n{doc.page_content.strip()}")
    return "\n\n".join(partes)

# --- Chain RAG (LCEL) ---
chain_rag = (
    {
        "context":  retriever | formatar_docs,
        "pergunta": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# --- Chain SEM RAG (para comparação) ---
prompt_sem_rag = ChatPromptTemplate.from_template("""
Você é um assistente de RH. Responda a pergunta abaixo.

Pergunta: {pergunta}
""")

chain_sem_rag = prompt_sem_rag | llm | StrOutputParser()

print("Chains prontas")

Chains prontas


## Comparação: Com RAG vs. Sem RAG

Fazemos a mesma pergunta nas duas chains e comparamos as respostas.

In [ ]:
# ── CÉLULA 5 · Demonstração ────────────────────────────────────────────────────

perguntas = [
    "Quantos dias de férias um funcionário da Nexus tem por ano?",
    "Qual o prazo para pedir reembolso de despesas?",
    "Qual o valor do auxílio home office?",
    "Quando ocorre o onboarding técnico para novos funcionários?",
]

separador = "-" * 70

for pergunta in perguntas:
    print(f"\n{'='*70}")
    print(f"{pergunta}")
    print(separador)

    resp_sem = chain_sem_rag.invoke({"pergunta": pergunta})
    print(f"SEM RAG:\n{resp_sem.strip()}")
    print(separador)

    resp_com = chain_rag.invoke(pergunta)
    print(f"COM RAG:\n{resp_com.strip()}")


Quantos dias de férias um funcionário da Nexus tem por ano?
----------------------------------------------------------------------
SEM RAG:
Olá! Como assistente de RH da Nexus, posso informar que os funcionários da empresa têm direito a 20 dias de férias por ano, mais 1 dia de férias por ano de serviço, conforme estabelecido na nossa política de benefícios. Além disso, os funcionários também têm direito a outros tipos de folgas, como feriados e dias de descanso remunerados.

Lembre-se de que essas informações podem variar dependendo da posição e do contrato de trabalho do funcionário. Se você tiver alguma dúvida ou precisar de mais informações, por favor, não hesite em perguntar!
----------------------------------------------------------------------
COM RAG:
De acordo com a Política de Férias – Nexus Tecnologia Ltda. (fonte: politica_ferias_v2.pdf), um funcionário da Nexus tem direito a 25 dias corridos de férias por ano completado.

Qual o prazo para pedir reembolso de despesas?
----

## Bônus: Inspecionar o que foi recuperado

É importante saber **quais chunks** o retriever trouxe para cada pergunta.
Sem isso, depurar um RAG que responde errado é muito difícil.

In [ ]:
# ── Inspecionar o retriever ────────────────────────────────────────

pergunta_teste = "Qual o limite de reembolso mensal?"

chunks_recuperados = retriever.invoke(pergunta_teste)

print(f"Pergunta: '{pergunta_teste}'")
print(f"→ {len(chunks_recuperados)} chunk(s) recuperado(s)\n")

for i, chunk in enumerate(chunks_recuperados):
    print(f"[Chunk {i+1}]")
    print(f"  Fonte      : {chunk.metadata['fonte']}")
    print(f"  Departamento: {chunk.metadata['departamento']}")
    print(f"  Conteúdo   : {chunk.page_content.strip()[:200]}...")
    print()

Pergunta: 'Qual o limite de reembolso mensal?'
→ 2 chunk(s) recuperado(s)

[Chunk 1]
  Fonte      : politica_reembolso_v3.pdf
  Departamento: financeiro
  Conteúdo   : Política de Reembolso de Despesas – Nexus Tecnologia Ltda.
        Despesas de viagem e representação devem ser solicitadas em até 10 dias
        úteis após o evento, via formulário REE-07 no portal ...

[Chunk 2]
  Fonte      : beneficios_2024.pdf
  Departamento: rh
  Conteúdo   : Benefícios – Nexus Tecnologia Ltda.
        Vale-refeição: R$ 45,00 por dia útil (cartão Alelo).
        Plano de saúde: Unimed Nacional, cobertura nacional, sem coparticipação.
        Auxílio home o...

